<a href="https://colab.research.google.com/github/Carlos608/rs_book/blob/main/cap4_pdi/cap4_assinaturas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Parte 1: Configuração do Ambiente e Bibliotecas

Configuração Obrigatória:
Para executar este notebook, você precisa de um projeto ativo no Google Cloud. Se você ainda não tem um, crie-o em console.cloud.google.com. Substitua o valor da variável project_id pelo seu ID pessoal.

In [ ]:
import ee
import geemap

# 1. Autenticação e Inicialização com o ID do Projeto
# No seu caso pessoal, o ID é 'ee-csouzajr'
project_id = 'ee-csouzajr'

try:
    # Tenta inicializar diretamente (funciona se já estiver autenticado na sessão do Colab)
    ee.Initialize(project=project_id)
    print(f"Sucesso: Earth Engine inicializado com o projeto: {project_id}")
except:
    # Se falhar, solicita a autenticação
    ee.Authenticate()
    ee.Initialize(project=project_id)
    print(f"Sucesso: Earth Engine autenticado e inicializado com o projeto: {project_id}")

2. Filtragem de Dados e Mosaico de Mediana
Trabalhar na Amazônia exige estratégia devido à alta nebulosidade. Em vez de usarmos uma única imagem, acessamos uma Coleção (ImageCollection) e aplicamos filtros de:

Espaço (filterBounds): Delimitamos nossa área de estudo.

Tempo (filterDate): Escolhemos o período seco (Junho a Outubro).

Qualidade (filter): Descartamos imagens com mais de 10% de nuvens.

O redutor .median() cria um mosaico onde cada píxel é a mediana de todos os píxeis limpos encontrados no período, eliminando sombras e ruídos.

In [ ]:
# 3. Extração de Assinaturas Espectrais (Versão Final Corrigida)
import pandas as pd
import matplotlib.pyplot as plt

# Definindo os nomes amigáveis para as bandas do Sentinel-2
# Precisamos que a ordem aqui seja a mesma da lista 'bands'
bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
labels = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']

# 1. Extrair os valores dos píxeis (GEE -> Python)
data = image.select(bands).reduceRegions(
    collection=points,
    reducer=ee.Reducer.mean(),
    scale=10
).getInfo()

# 2. Organizando os dados em uma tabela (DataFrame)
rows = []
for feature in data['features']:
    label_name = feature['properties']['label']
    # Extrai os valores das bandas para este ponto específico
    values = [feature['properties'][b] for b in bands]
    rows.append({'Alvo': label_name, **dict(zip(labels, values))})

df_plot = pd.DataFrame(rows).set_index('Alvo').T

# 3. Gerando o Gráfico com Matplotlib
plt.figure(figsize=(10, 6))
for column in df_plot.columns:
    plt.plot(df_plot.index, df_plot[column], marker='o', label=column, linewidth=2)

plt.title('Assinaturas Espectrais: Floresta vs Solo Exposto (Sentinel-2)')
plt.xlabel('Bandas Espectrais (Comprimento de Onda)')
plt.ylabel('Reflectância de Superfície (0 a 1)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

4. Visualização no Mapa
Por fim, é crucial ver onde esses pontos estão localizados para validar nossa análise. Usaremos uma composição de Falsa Cor (NIR-R-G), onde a vegetação aparece em tons de vermelho vibrante, facilitando a distinção de áreas degradadas.

In [ ]:
Map = geemap.Map()
Map.centerObject(roi, 13)

# Parâmetros de visualização em Falsa Cor (B8, B4, B3)
vis_params = {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 0.4}

Map.addLayer(image, vis_params, 'Composição Falsa Cor (NIR-R-G)')
Map.addLayer(points, {}, 'Pontos de Coleta')
Map